# Web Scraper — Coleta Profunda de Conteúdo Central, Links e Figuras

Este notebook lê uma **lista de páginas semente** e percorre **todos os níveis de links** que conseguir
alcançar a partir delas (crawl em profundidade ilimitada, com salvaguardas), restrito ao mesmo domínio.

Diferenças em relação à versão anterior:

- **Profundidade ilimitada** (`PROFUNDIDADE_MAX = None`): segue links até esgotar o que houver no domínio,
  com limites de segurança (`MAX_PAGINAS`) para não rodar indefinidamente.
- **Foco no conteúdo central**: o scraper tenta isolar a *região principal* da página
  (`<main>`, `<article>`, `role=main`...) e **descarta** cabeçalho, rodapé, menus de navegação,
  menus suspensos, barras laterais e demais elementos repetidos de layout. Assim, tanto o **texto**
  quanto os **links seguidos** vêm do miolo da página, não dos menus.
- **Figuras do conteúdo central**: imagens e diagramas (`<img>`, `<figure>`, `<svg>`) que estão
  dentro do conteúdo principal são coletados, com URL absoluta, texto alternativo e legenda.

No final, salva tudo em um arquivo **JSON** onde cada registro contém:

| Campo | Descrição |
|-------|-----------|
| `url` | endereço da página |
| `titulo` | título (`<title>` ou `<h1>`) |
| `texto` | texto visível **do conteúdo central** |
| `relacao` | `'raiz'` (semente) ou `'filho'` (descoberto a partir de outra) |
| `pai` | URL da página de onde este link foi descoberto (`null` para raízes) |
| `profundidade` | nível no crawl (0 = semente) |
| `links` | links **do conteúdo central** encontrados na página |
| `figuras` | imagens/diagramas do conteúdo central (`url`, `alt`, `legenda`, `tipo`) |

> **Boas práticas:** o scraper respeita um intervalo entre requisições, define um *User-Agent* e checa o
> `robots.txt`. Use sempre de forma responsável e de acordo com os termos do site.


## 1. Instalação e importação das bibliotecas

- **requests**: baixar o HTML das páginas.
- **beautifulsoup4**: analisar (*parse*) o HTML e extrair texto, links e figuras.

Se estiver no Google Colab, a célula abaixo já instala o que for necessário.

In [1]:
# Em ambientes onde as libs não existam (ex.: Colab), descomente:
# !pip install requests beautifulsoup4 openpyxl selenium undetected-chromedriver
#
# openpyxl  -> ler a planilha .xlsx local (seção 2.1)
# selenium / undetected-chromedriver -> coleta dinâmica e menos detectável (seção 3.5)
#              No Linux com Google Chrome instalado, o Selenium 4.6+ baixa o driver sozinho.

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import time
import json
import os, json, shutil, time
from collections import deque, Counter


print('Bibliotecas importadas com sucesso!')


Bibliotecas importadas com sucesso!


## 2. Configuração

Aqui definimos:
- a **lista de páginas semente** (as URLs iniciais);
- a **profundidade máxima** — agora `None` significa **ilimitada** (percorre tudo o que alcançar);
- um **teto de páginas** (`MAX_PAGINAS`) como salvaguarda para crawls muito grandes;
- se devemos **restringir ao mesmo domínio**;
- parâmetros de educação com o servidor (*delay*, *timeout*, *User-Agent*).

Para analisar outras páginas, basta editar a lista `PAGINAS_SEMENTE`.

Segue um exemplo de "robots.txt":

User-agent: *

Disallow: /admin/

Disallow: /carrinho/

Allow: /produtos/

Crawl-delay: 10

Sitemap: https://exemplo.com/sitemap.xml


In [2]:
# No Google Colab, monta o Drive para salvar lá. Localmente, isto é ignorado.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    print('Não estou no Colab — os arquivos serão salvos no diretório local.')


Não estou no Colab — os arquivos serão salvos no diretório local.


In [3]:
# Lista de páginas iniciais (você pode colocar quantas quiser)
PAGINAS_SEMENTE = [
   "https://www.iso.org/standard/56641.html"
]

# 'https://sbis.org.br/certificacoes/certificacao-de-ia/'

# Profundidade do crawl:
#   None = ILIMITADA (segue todos os níveis alcançáveis no domínio) -- CUIDADO: muito lento
#   0    = só as sementes | 1 = sementes + links diretos | 2 = ... e assim por diante
#
# Com VÁRIAS sementes de sites grandes (eur-lex, unesco, iso...), 'ilimitada' fica
# praticamente infinito. Profundidade 1 dá boa cobertura sem explodir o tempo.
PROFUNDIDADE_MAX = 1

MESMO_DOMINIO    = True     # True = só segue links do mesmo domínio
MAX_PAGINAS      = 3000      # salvaguarda GLOBAL: para o crawl após N páginas no total
MAX_POR_DOMINIO  = 1000       # salvaguarda POR SITE: evita que um domínio gigante consuma tudo
DELAY_SEGUNDOS   = 0.5      # pausa entre requisições (educação com o servidor)
TIMEOUT          = 6        # tempo máximo de espera por página (segundos) -- 6s já cobre sites OK
RESPEITAR_ROBOTS = True     # checar robots.txt antes de baixar

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (compatible; ScraperDidatico/1.0; '
        '+https://example.org/bot-info)'
    )
}

# Destino do JSON — AGNÓSTICO ao ambiente:
#   - no Google Colab (com Drive montado): salva no seu Drive;
#   - localmente: salva no diretório atual.
if os.path.isdir('/content/drive/MyDrive'):
    ARQUIVO_SAIDA = '/content/drive/MyDrive/dataset_selecao_paginas_web/conteudo_coletado.json'
else:
    ARQUIVO_SAIDA = 'conteudo_coletado.json'

# CSV onde as páginas que falharam (estático e dinâmico) são registradas para reprocessar.
ARQUIVO_ERROS = 'paginas_com_erro.csv'
# CSV separado para URLs válidas que foram PULADAS por limite (cota de domínio ou
# teto global). Não são erros de coleta — ficam aqui para você poder revisá-las.
ARQUIVO_IGNORADAS = 'paginas_ignoradas_por_limite.csv'

# A cada quantas páginas sincronizar com o destino final (Drive).
# Durante o crawl, gravamos rápido num arquivo LOCAL e copiamos para o Drive
# a cada INTERVALO_SALVAR páginas (e no fim). Isso evita reescrever no Drive a cada página.
INTERVALO_SALVAR = 10

# Cria a pasta de destino se ela ainda não existir (evita erro ao salvar).
# Se for usar o Drive, monte-o antes:  from google.colab import drive; drive.mount('/content/drive')
import os
_dir_saida = os.path.dirname(ARQUIVO_SAIDA)
if _dir_saida:
    os.makedirs(_dir_saida, exist_ok=True)

_prof_txt = 'ilimitada' if PROFUNDIDADE_MAX is None else PROFUNDIDADE_MAX
print(f'{len(PAGINAS_SEMENTE)} página(s) semente configurada(s).')
print(f'Profundidade máxima: {_prof_txt} | Mesmo domínio: {MESMO_DOMINIO}')
print(f'Tetos: {MAX_PAGINAS} páginas no total, {MAX_POR_DOMINIO} por domínio | Delay: {DELAY_SEGUNDOS}s')

1 página(s) semente configurada(s).
Profundidade máxima: 1 | Mesmo domínio: True
Tetos: 3000 páginas no total, 1000 por domínio | Delay: 0.5s


### 2.1 Montar `PAGINAS_SEMENTE` a partir da planilha de metadados

Em vez de digitar as sementes à mão, este bloco lê a **planilha do corpus** e usa como sementes
apenas as linhas cuja coluna **`source_type`** vale **`html`** — descartando `pdf` e linhas sem tipo.
A URL vem da coluna **`source_url`**.

**Leitura agnóstica ao ambiente.** O bloco tenta, nesta ordem:

1. um arquivo **`.xlsx` local** (ex.: `Corpus_metadados_exemplo.xlsx`), se existir no diretório;
2. caso não exista, a **planilha do Google Sheets** publicada por link (exportação CSV).

Assim o mesmo notebook funciona rodando **localmente** (lendo o `.xlsx` ao lado) ou no **Colab**
(lendo o Sheets online), sem precisar trocar código. Para o Google Sheets, a planilha precisa estar
compartilhada como *Qualquer pessoa com o link → Leitor*.


In [4]:
import csv
import io
import re
import os

# === Origem da planilha ============================================================
# O bloco é AGNÓSTICO: usa o .xlsx local se existir; senão, cai no Google Sheets online.

# Caminho do .xlsx local (procurado primeiro). Ajuste se o arquivo tiver outro nome/lugar.
ARQUIVO_PLANILHA_LOCAL = 'Corpus_metadados_exemplo.xlsx'

# Link do Google Sheets (usado só se o .xlsx local NÃO for encontrado).
URL_PLANILHA = 'https://docs.google.com/spreadsheets/d/1BzRxU1Ono6e4h8lH5so8c6H1HjA69rzG/edit?usp=sharing&ouid=105965451269116951803&rtpof=true&sd=true'

# Opcional: gid da aba específica (número após 'gid=' na URL). None = primeira aba.
GID_ABA = None

# === Regra de seleção ==============================================================
COLUNA_URL   = 'source_url'    # coluna com o endereço
COLUNA_TIPO  = 'source_type'   # coluna que decide se coletamos
TIPO_DESEJADO = 'html'         # coleta a linha somente se source_type == 'html'


def _url_para_csv_export(url, gid=None):
    """Converte um link de Google Sheets na URL de exportação CSV."""
    m = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', url)
    if not m:
        raise ValueError('Não parece um link de Google Sheets válido: ' + url)
    sheet_id = m.group(1)
    if gid is None:
        g = re.search(r'[#&?]gid=([0-9]+)', url)
        gid = g.group(1) if g else None
    base = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'
    if gid is not None:
        base += f'&gid={gid}'
    return base


def _linhas_do_xlsx(caminho):
    """Lê a 1ª aba de um .xlsx e devolve lista de dicts {coluna: valor}."""
    import openpyxl  # instale com: pip install openpyxl
    wb = openpyxl.load_workbook(caminho, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    linhas_raw = list(ws.iter_rows(values_only=True))
    if not linhas_raw:
        return []
    hdr = [str(c).strip() if c is not None else '' for c in linhas_raw[0]]
    saida = []
    for r in linhas_raw[1:]:
        d = {col: (r[i] if i < len(r) else None) for i, col in enumerate(hdr)}
        saida.append(d)
    return saida


def _linhas_do_sheets(url_planilha, gid=None):
    """Lê a planilha publicada (CSV) e devolve lista de dicts {coluna: valor}."""
    csv_url = _url_para_csv_export(url_planilha, gid)
    resp = requests.get(csv_url, headers=HEADERS, timeout=TIMEOUT)
    resp.raise_for_status()
    tipo = resp.headers.get('Content-Type', '')
    if 'text/csv' not in tipo and '<html' in resp.text[:200].lower():
        raise PermissionError(
            'A planilha não retornou CSV. Verifique se está compartilhada como '
            '"Qualquer pessoa com o link pode ver".'
        )
    leitor = csv.DictReader(io.StringIO(resp.text))
    return list(leitor)


def carregar_linhas_planilha():
    """Decide a origem: .xlsx local (se existir) ou Google Sheets online."""
    if ARQUIVO_PLANILHA_LOCAL and os.path.exists(ARQUIVO_PLANILHA_LOCAL):
        print(f'[planilha] lendo arquivo local: {ARQUIVO_PLANILHA_LOCAL}')
        return _linhas_do_xlsx(ARQUIVO_PLANILHA_LOCAL)
    print('[planilha] arquivo local não encontrado; lendo do Google Sheets online.')
    return _linhas_do_sheets(URL_PLANILHA, GID_ABA)


def ler_sementes_da_planilha(coluna_url=COLUNA_URL, coluna_tipo=COLUNA_TIPO,
                             tipo_desejado=TIPO_DESEJADO):
    """Carrega a planilha e retorna as URLs cujo `source_type` é o desejado ('html').

    Regra (INCLUSIVA): a linha entra apenas se source_type == tipo_desejado.
    Linhas sem URL são ignoradas; linhas com URL mas de outro tipo (pdf, vazio)
    são contadas como descartadas.
    """
    linhas = carregar_linhas_planilha()
    if linhas:
        cols = list(linhas[0].keys())
        for c in (coluna_url, coluna_tipo):
            if c not in cols:
                raise KeyError(f'Coluna {c!r} não encontrada. Colunas disponíveis: {cols}')

    sementes, vistos = [], set()
    sem_url = desc_tipo = 0
    for d in linhas:
        url = (str(d.get(coluna_url) or '')).strip()
        tipo = (str(d.get(coluna_tipo) or '')).strip().lower()
        if not url:
            sem_url += 1
            continue
        if tipo != tipo_desejado.lower():
            desc_tipo += 1
            continue
        if url not in vistos:
            vistos.add(url)
            sementes.append(url)

    print(f'\nSementes com source_type == {tipo_desejado!r}: {len(sementes)}')
    print(f'Descartadas por tipo diferente (ex.: pdf): {desc_tipo}')
    print(f'Linhas sem URL ignoradas: {sem_url}')
    return sementes


# Monta a lista de sementes a partir da planilha.
# (Se preferir manter as sementes manuais da célula anterior, comente a linha abaixo.)
PAGINAS_SEMENTE = ler_sementes_da_planilha()

print(f'\nPAGINAS_SEMENTE agora tem {len(PAGINAS_SEMENTE)} item(ns).')
for s in PAGINAS_SEMENTE[:10]:
    print('  -', s)
if len(PAGINAS_SEMENTE) > 10:
    print(f'  ... e mais {len(PAGINAS_SEMENTE) - 10}.')


[planilha] arquivo local não encontrado; lendo do Google Sheets online.

Sementes com source_type == 'html': 48
Descartadas por tipo diferente (ex.: pdf): 42
Linhas sem URL ignoradas: 9

PAGINAS_SEMENTE agora tem 48 item(ns).
  - https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689
  - https://www.iso.org/standard/56641.html
  - https://www.iso.org/standard/74296.html
  - https://www.iso.org/standard/78507.html
  - https://www.iso.org/standard/42001
  - https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13709.htm
  - https://oecd.ai/en/ai-principles
  - https://www.gov.br/anpd/pt-br/acesso-a-informacao/institucional/atos-normativos/regulamentacoes_anpd/resolucao-cd-anpd-no-2-de-27-de-janeiro-de-2022
  - https://ico.org.uk/for-organisations/uk-gdpr-guidance-and-resources/accountability-and-governance/data-protection-impact-assessments-dpias/what-is-a-dpia/#what1
  - https://www.planalto.gov.br/ccivil_03/leis/2002/l10406compilada.htm
  ... e mais 38.


## 3. Funções auxiliares

Vamos quebrar o problema em funções pequenas e fáceis de entender. Cada uma faz **uma** coisa.

### 3.1 Normalizar URLs e comparar domínios

Links em uma página podem ser **relativos** (`/contato`) ou conter âncoras (`#secao`). Precisamos
transformá-los em URLs absolutas e comparáveis.

In [5]:
def _fragmento_significativo(frag):
    """Um fragmento (#...) é 'significativo' quando IDENTIFICA o documento, em vez
    de apontar para uma seção de rolagem na mesma página.

    Alguns sites (ex.: a plataforma OBP da ISO) carregam o conteúdo via JavaScript
    e usam o trecho após o '#' como rota:  .../obp/ui/en/#iso:std:iso-iec:38507:...
    Se removêssemos esse fragmento (como fazíamos antes), TODAS as normas ISO
    colapsariam na mesma URL e o crawler trataria a 2ª em diante como 'já visitada'.

    Heurística: é significativo se contém ':' (rota tipo iso:std:...), ou começa
    com '!' ou '/' (padrões de hash-routing como #!/rota ou #/rota).
    """
    if not frag:
        return False
    f = frag.strip()
    if ':' in f:
        return True
    if f.startswith('!') or f.startswith('/'):
        return True
    return False


def normalizar_url(base, link):
    """Converte um link (relativo ou absoluto) em URL absoluta e limpa.

    - Resolve links relativos usando a URL da página atual (base).
    - Remove a âncora (#secao) quando ela é só rolagem na mesma página.
    - MANTÉM o fragmento quando ele identifica o documento (ex.: #iso:std:...),
      pois nesses sites o '#' faz parte da identidade da página.
    """
    if not link:
        return None
    url = urljoin(base, link.strip())
    parsed = urlparse(url)
    # Mantém só http/https (ignora mailto:, tel:, javascript: etc.)
    if parsed.scheme not in ('http', 'https'):
        return None
    if _fragmento_significativo(parsed.fragment):
        return parsed.geturl()                       # preserva o fragmento
    return parsed._replace(fragment='').geturl()     # remove âncora de rolagem


def mesmo_dominio(url_a, url_b):
    """Retorna True se as duas URLs pertencem ao mesmo domínio (host)."""
    return urlparse(url_a).netloc.lower() == urlparse(url_b).netloc.lower()


# Teste rápido
base = 'https://sbis.org.br/certificacoes/certificacao-de-ia/'
print(normalizar_url(base, '/contato'))
print(normalizar_url(base, 'https://outrosite.com/pagina'))
print(normalizar_url(base, '#secao-x'), '(âncora de rolagem -> removida)')
iso = 'https://www.iso.org/obp/ui/en/'
print(normalizar_url(iso, '#iso:std:iso-iec:38507:ed-1:v1:en'), '(fragmento mantido)')
print(mesmo_dominio(base, 'https://sbis.org.br/certificacoes/certificacao-de-ia/dimensoes-e-dominios-avaliados/'))


https://sbis.org.br/contato
https://outrosite.com/pagina
https://sbis.org.br/certificacoes/certificacao-de-ia/ (âncora de rolagem -> removida)
https://www.iso.org/obp/ui/en/#iso:std:iso-iec:38507:ed-1:v1:en (fragmento mantido)
True


### 3.2 Verificar permissão no `robots.txt`

O arquivo `robots.txt` informa quais áreas do site os robôs podem ou não acessar. É boa prática (e às
vezes obrigação legal/contratual) respeitá-lo. Guardamos um *cache* por domínio para não baixá-lo toda vez.

In [6]:
_robots_cache = {}  # cache: dominio -> RobotFileParser (ou None se indisponível)

def pode_acessar(url, respeitar_robots=None):
    """Consulta o robots.txt do domínio e diz se o nosso bot pode acessar a URL.

    `respeitar_robots`: se None, usa a variável global RESPEITAR_ROBOTS.
    Aceitar este 2º argumento mantém a função compatível com a chamada feita
    dentro de coletar(), que passa pode_acessar(url, respeitar_robots).

    A leitura do robots.txt é feita com `requests` (mesmo User-Agent e timeout
    do resto do scraper), em vez de deixar o RobotFileParser fazer a requisição
    sozinho. Assim evitamos travas em sites lentos e ficamos consistentes.
    """
    if respeitar_robots is None:
        respeitar_robots = RESPEITAR_ROBOTS
    if not respeitar_robots:
        return True
    parsed = urlparse(url)
    dominio = f'{parsed.scheme}://{parsed.netloc}'
    if dominio not in _robots_cache:
        rp = RobotFileParser()
        robots_url = urljoin(dominio, '/robots.txt')
        try:
            resp = requests.get(robots_url, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code >= 400:
                # 404/403 etc.: sem robots.txt utilizável -> tratamos como liberado
                rp = None
            else:
                rp.parse(resp.text.splitlines())
        except requests.RequestException:
            # Falha de rede ao ler o robots.txt: seguimos com cautela (permitido)
            rp = None
        _robots_cache[dominio] = rp
    rp = _robots_cache[dominio]
    if rp is None:
        return True
    return rp.can_fetch(HEADERS['User-Agent'], url)

print('robots.txt permite acessar a semente?',
      pode_acessar(PAGINAS_SEMENTE[0]))
      #pode_acessar('https://unesdoc.unesco.org/ark:/48223/pf0000386276'))

robots.txt permite acessar a semente? True


### 3.3 Isolar o **conteúdo central** da página

Esta é a peça nova mais importante. Antes de extrair texto, links ou figuras, tentamos localizar a
**região principal** da página e jogar fora o que é "moldura" repetida do site.

A estratégia tem duas etapas:

1. **Remover** elementos estruturais de navegação/layout: `<header>`, `<footer>`, `<nav>`, `<aside>`,
   elementos com `role` de navegação/banner, e blocos cujo `id`/`class` sugerem menu, *dropdown*,
   *sidebar*, *breadcrumb*, *cookie*, *social*, etc.
2. **Selecionar** o melhor candidato a conteúdo principal — em ordem de preferência: `<main>`,
   `[role=main]`, `<article>`, ou contêineres comuns (`#content`, `.entry-content`, `.post-content`...).
   Se nada for encontrado, caímos de volta no `<body>` já limpo.

Assim, evitamos coletar links de menu suspenso e textos repetidos, mantendo o miolo informativo.

In [7]:
# Padrões que, aparecendo em id/class, indicam 'moldura' (não é conteúdo central).
#
# IMPORTANTE: usamos padrões ESPECÍFICOS, não substrings curtas e genéricas.
# Substrings como 'menu' ou 'sidebar' são perigosas: além da navegação global
# do site, elas casam com BLOCOS DE CONTEÚDO legítimos. No tema deste site, por
# exemplo, 'tekup-service-menu' e 'tekup-pd-sidebar' são uma coluna de links para
# subpáginas (conteúdo!), não a moldura do site. Por isso preferimos termos como
# 'navbar', 'main-menu', 'nav-menu', 'site-header' a apenas 'menu'/'header'.
_LIXO_PADROES = (
    'navbar', 'main-menu', 'nav-menu', 'primary-menu', 'menu-principal',
    'dropdown', 'submenu', 'megamenu', 'offcanvas',
    'site-header', 'page-header', 'masthead', 'topbar',
    'site-footer', 'page-footer', 'rodape',
    'breadcrumb', 'cookie', 'consent', 'social-', 'share-', 'compartilh',
    'newsletter', 'skip-link', 'modal', 'popup',
    'pagination', 'paginacao', 'search-form', 'sr-only', 'screen-reader',
)

# Seletores preferidos para o conteúdo principal, em ordem de prioridade.
_SELETORES_PRINCIPAIS = (
    'main',
    '[role="main"]',
    'article',
    '#content', '#main', '#primary',
    '.entry-content', '.post-content', '.page-content',
    '.content-area', '.site-content', '.main-content', '.elementor-section-wrap',
)


def _parece_lixo(tag):
    """True se o id/class da tag sugere navegação/layout (não conteúdo).

    Aceita só tags-elemento reais; nós de texto/comentário não têm .get().
    """
    if not hasattr(tag, 'get'):
        return False
    ident = ' '.join(filter(None, [
        tag.get('id', '') or '',
        ' '.join(tag.get('class', []) or []),
        tag.get('role', '') or '',
    ])).lower()
    if not ident:
        return False
    return any(p in ident for p in _LIXO_PADROES)


def isolar_conteudo_central(soup):
    """Retorna um BeautifulSoup/Tag contendo só o conteúdo central da página.

    Trabalha sobre uma CÓPIA para não destruir o soup original.
    """
    # Trabalha numa cópia independente
    soup = BeautifulSoup(str(soup), 'html.parser')

    # 1) Remove ruído técnico
    for tag in soup(['script', 'style', 'noscript', 'template', 'form', 'iframe']):
        tag.decompose()

    # 2) Remove blocos estruturais de navegação/layout
    for tag in soup.find_all(['header', 'footer', 'nav', 'aside']):
        tag.decompose()
    for tag in soup.find_all(attrs={'role': ['navigation', 'banner', 'contentinfo', 'search', 'menu', 'menubar']}):
        tag.decompose()

    # 3) Remove blocos cujo id/class indicam moldura (menus suspensos, sidebars, etc.)
    #    Materializamos a lista e checamos se o nó ainda está na árvore, pois um
    #    decompose() pode desconectar descendentes antes de o loop chegar neles.
    for tag in list(soup.find_all(True)):
        if tag.parent is None:
            continue  # já foi removido junto com um ancestral
        if _parece_lixo(tag):
            tag.decompose()

    # 4) Escolhe o melhor candidato a conteúdo principal
    for seletor in _SELETORES_PRINCIPAIS:
        alvo = soup.select_one(seletor)
        if alvo and alvo.get_text(strip=True):
            return alvo

    # 5) Fallback: body limpo (ou o próprio soup)
    return soup.body or soup


print('Função isolar_conteudo_central() pronta.')

Função isolar_conteudo_central() pronta.


### 3.4 Extrair título, texto, links e **figuras** do conteúdo central

Com a região principal isolada, extraímos:
- **título** (`<title>`, com fallback para o primeiro `<h1>`) — do documento inteiro;
- **texto visível** — apenas do conteúdo central;
- **links** (`<a href=...>`) — apenas do conteúdo central, já normalizados;
- **figuras** — `<img>`, `<svg>` e `<figure>` dentro do conteúdo central, com URL absoluta,
  texto alternativo (`alt`) e legenda (`<figcaption>`) quando houver.

Para imagens, resolvemos também `srcset` e atributos *lazy-load* comuns (`data-src`).

In [8]:
def extrair_titulo(soup):
    """Pega o <title>; se não houver, tenta o primeiro <h1>. Usa o documento inteiro."""
    if soup.title and soup.title.string:
        return soup.title.string.strip()
    h1 = soup.find('h1')
    if h1:
        return h1.get_text(strip=True)
    return '(sem título)'


def extrair_texto(central):
    """Extrai o texto visível do conteúdo central já isolado."""
    texto = central.get_text(separator=' ', strip=True)
    return ' '.join(texto.split())


def extrair_links(central, url_base):
    """Coleta os links <a href> do conteúdo central, como URLs absolutas e únicas."""
    links = set()
    for a in central.find_all('a', href=True):
        url = normalizar_url(url_base, a['href'])
        if url:
            links.add(url)
    return sorted(links)


def _melhor_src(img):
    """Descobre a melhor URL de uma <img>, considerando lazy-load e srcset."""
    # Atributos comuns de lazy-load primeiro
    for attr in ('src', 'data-src', 'data-original', 'data-lazy-src'):
        val = img.get(attr)
        if val and not val.startswith('data:'):
            return val
    # srcset: pega a última (geralmente a maior) entrada
    srcset = img.get('srcset') or img.get('data-srcset')
    if srcset:
        candidatos = [p.strip().split(' ')[0] for p in srcset.split(',') if p.strip()]
        if candidatos:
            return candidatos[-1]
    return None


def extrair_figuras(central, url_base):
    """Coleta imagens/diagramas do conteúdo central.

    Retorna lista de dicts: {url, alt, legenda, tipo}.
    - <img>  -> tipo 'imagem' (inclui diagramas exportados como PNG/JPG/SVG)
    - <svg>  -> tipo 'svg'    (diagramas vetoriais inline; pode não ter URL)
    """
    figuras = []
    vistos = set()

    # Legendas: mapeia uma <figure> à sua <figcaption>
    def legenda_de(tag):
        fig = tag.find_parent('figure')
        if fig:
            cap = fig.find('figcaption')
            if cap:
                return cap.get_text(' ', strip=True)
        return ''

    # 1) Imagens <img>
    for img in central.find_all('img'):
        src = _melhor_src(img)
        url = normalizar_url(url_base, src) if src else None
        if not url or url in vistos:
            continue
        vistos.add(url)
        figuras.append({
            'url':     url,
            'alt':     (img.get('alt') or '').strip(),
            'legenda': legenda_de(img),
            'tipo':    'imagem',
        })

    # 2) Diagramas vetoriais inline <svg> (sem arquivo próprio)
    for svg in central.find_all('svg'):
        # Tenta um rótulo: <title>/<desc> dentro do svg, ou aria-label
        rotulo = ''
        t = svg.find('title')
        if t and t.get_text(strip=True):
            rotulo = t.get_text(strip=True)
        elif svg.get('aria-label'):
            rotulo = svg.get('aria-label').strip()
        figuras.append({
            'url':     None,            # SVG inline não tem URL própria
            'alt':     rotulo,
            'legenda': legenda_de(svg),
            'tipo':    'svg',
        })

    return figuras


def _extrair_de_html(html, url):
    """Dado o HTML (cru ou renderizado), isola o conteúdo central e extrai tudo."""
    soup = BeautifulSoup(html, 'html.parser')
    central = isolar_conteudo_central(soup)
    return {
        'titulo':  extrair_titulo(soup),
        'texto':   extrair_texto(central),
        'links':   extrair_links(central, url),
        'figuras': extrair_figuras(central, url),
    }


# Limiar (nº de caracteres de texto útil) abaixo do qual consideramos que a
# coleta estática "falhou" — sinal típico de página renderizada por JavaScript.
MIN_TEXTO_UTIL = 200


def _conteudo_vazio(dados):
    """True se o resultado estático veio vazio/quase vazio (provável página JS)."""
    if dados is None:
        return True
    return len((dados.get('texto') or '').strip()) < MIN_TEXTO_UTIL


# --- Guarda de extensão: o fallback dinâmico só faz sentido para HTML ------------
# O Chrome renderizaria (ou baixaria) arquivos como .rtf/.doc/.zip de forma inútil,
# às vezes exibindo a última página que estava na tela. Barramos essas extensões.
_EXT_NAO_HTML = (
    '.pdf', '.rtf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx',
    '.zip', '.rar', '.7z', '.gz', '.tar', '.csv', '.txt', '.odt', '.ods',
    '.jpg', '.jpeg', '.png', '.gif', '.svg', '.mp4', '.mp3', '.avi',
)


def _extensao_nao_html(url):
    """True se a URL aponta claramente para um arquivo que não é página HTML."""
    caminho = urlparse(url).path.lower()
    return caminho.endswith(_EXT_NAO_HTML)


# --- Detecção de páginas de bloqueio / desafio anti-bot -----------------------
# Serviços como o Cloudflare intercalam uma página "Just a moment..." /
# "Um momento..." que EXECUTA JavaScript e TEM texto (às vezes >200 caracteres),
# então passaria despercebida pelo limiar de _conteudo_vazio(). Precisamos
# reconhecê-la explicitamente para NÃO salvar o desafio no lugar do conteúdo.
_SINAIS_BLOQUEIO_TITULO = (
    'just a moment', 'um momento', 'attention required', 'acesso negado',
    'access denied', 'checking your browser', 'verifying you are human',
)
_SINAIS_BLOQUEIO_TEXTO = (
    'verificação de segurança', 'checking if the site connection is secure',
    'proteção contra bots', 'protection against malicious bots',
    'enable javascript and cookies to continue', 'performance & security by cloudflare',
    'executando verificação de segurança', 'esperando a resposta de',
    'ray id', 'cf-ray', 'ddos protection by',
)


def _parece_bloqueio(dados):
    """True se o dict extraído é, na verdade, uma página de desafio/bloqueio anti-bot."""
    if not dados:
        return False
    titulo = (dados.get('titulo') or '').strip().lower()
    texto = (dados.get('texto') or '').strip().lower()
    if any(sig in titulo for sig in _SINAIS_BLOQUEIO_TITULO):
        return True
    if any(sig in texto for sig in _SINAIS_BLOQUEIO_TEXTO):
        return True
    # Heurística extra: texto muito curto E sem nenhum link é típico de interstício.
    if len(texto) < 400 and not dados.get('links'):
        # só marca como bloqueio se também houver pista de "aguarde/verificando"
        if any(p in texto for p in ('aguarde', 'wait', 'verific', 'moment')):
            return True
    return False


def baixar_pagina(url):
    """Baixa a página (estático) e devolve dict(titulo, texto, links, figuras, origem).

    Retorna None só quando nem o conteúdo serve E não há fallback aplicável.
    O acionamento do fallback dinâmico é decidido em coletar(), não aqui — esta
    função apenas reporta se o estático foi suficiente (via campo 'origem').
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f'  [ERRO estático] {url}: {e}')
        return None

    tipo = resp.headers.get('Content-Type', '')
    if 'text/html' not in tipo:
        print(f'  [PULADO] {url} não é HTML (Content-Type: {tipo})')
        return None

    dados = _extrair_de_html(resp.text, url)
    dados['origem'] = 'estatico'
    return dados


print('Funções de extração prontas.')


Funções de extração prontas.


### 3.5 Coleta dinâmica (JavaScript) — *fallback* menos detectável + detecção de bloqueio

Algumas páginas não trazem o conteúdo no HTML cru: ele é montado por **JavaScript**. Pior, sites como
a plataforma **OBP da ISO** ficam atrás do **desafio anti-bot da Cloudflare** (a página *"Just a
moment…" / "Um momento…"*), que roda quebra-cabeças de JavaScript para barrar automação. Um Chrome
headless "cru" costuma ser detectado e recebe só essa página de verificação — que tem texto e passaria
batido pelo limiar de vazio.

Duas defesas, portanto:

1. **Motor menos detectável.** Usamos o **`undetected-chromedriver`**, que mascara as flags de
   automação que a Cloudflare procura, e **esperamos** o desafio se resolver sozinho (ele costuma
   liberar após alguns segundos). Isso serve apenas para acessar a **prévia pública** da norma
   (título, escopo, introdução, termos) — material que a própria ISO disponibiliza gratuitamente.
2. **Detecção de bloqueio** (seção anterior): se, mesmo assim, o que voltar for a página de desafio,
   ela **não** é salva; a URL vai para o `paginas_com_erro.csv` com motivo `bloqueio_anti_bot`.

> **Escopo e limites.** Coletamos só a prévia pública (não o texto pago da norma) e mantemos um
> *delay* educado. Passar por *paywalls* ou controle de acesso de conteúdo licenciado **não** é
> objetivo aqui. Se a Cloudflare insistir em bloquear, a página simplesmente fica registrada como erro.

**Instalação (Linux com Google Chrome):**

```bash
pip install undetected-chromedriver selenium
```


In [9]:
# Caminho do binário do Chrome (None = deixa o driver localizar sozinho).
CHROME_BIN = None

# Segundos a esperar pela renderização / resolução do desafio anti-bot.
DYN_TIMEOUT = 30

# Tentativas de recarregar enquanto a página de desafio não sai.
DYN_MAX_TENTATIVAS = 3

# --- Ajustes para SPAs (apps JavaScript que montam o conteúdo após carregar) ---
# Timeout MAIOR só para renderização de conteúdo (SPAs podem demorar mais que o
# desafio anti-bot). Fica separado de DYN_TIMEOUT.
DYN_TIMEOUT_SPA = 45
# Nº de rolagens até o fim da página para disparar lazy-loading comum em SPAs.
DYN_SCROLLS = 4
# Pausa (s) entre cada poll de texto / rolagem.
DYN_POLL_INTERVALO = 0.5

_driver = None
_driver_indisponivel = False
_driver_tipo = None   # 'undetected' ou 'selenium'


def _criar_driver():
    """Cria um Chrome o menos detectável possível. Prefere undetected-chromedriver."""
    # 1ª opção: undetected-chromedriver (mascara flags de automação)
    try:
        import undetected_chromedriver as uc
        opts = uc.ChromeOptions()
        opts.add_argument('--no-sandbox')
        opts.add_argument('--disable-dev-shm-usage')
        opts.add_argument('--window-size=1366,2000')
        if CHROME_BIN:
            opts.binary_location = CHROME_BIN
        # headless=False tende a passar mais fácil pela Cloudflare; use True se não tiver display.
        drv = uc.Chrome(options=opts, headless=True)
        drv.set_page_load_timeout(DYN_TIMEOUT + 15)
        print('[dinâmico] undetected-chromedriver iniciado.')
        return drv, 'undetected'
    except Exception as e:
        print(f'[dinâmico] undetected-chromedriver indisponível ({type(e).__name__}). '
              f'Tentando Selenium comum...')

    # 2ª opção: Selenium comum (mais facilmente detectado, mas melhor que nada)
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument('--window-size=1366,2000')
    opts.add_argument(f"--user-agent={HEADERS['User-Agent']}")
    if CHROME_BIN:
        opts.binary_location = CHROME_BIN
    drv = webdriver.Chrome(options=opts)
    drv.set_page_load_timeout(DYN_TIMEOUT + 15)
    print('[dinâmico] Selenium Chrome comum iniciado.')
    return drv, 'selenium'


def _get_driver():
    """Cria (uma vez) e devolve o driver. None se indisponível."""
    global _driver, _driver_indisponivel, _driver_tipo
    if _driver is not None:
        return _driver
    if _driver_indisponivel:
        return None
    try:
        _driver, _driver_tipo = _criar_driver()
        return _driver
    except Exception as e:
        _driver_indisponivel = True
        print(f'[dinâmico] indisponível ({type(e).__name__}: {e}). '
              f'Páginas que precisarem de JS irão para o CSV de erros.')
        return None


def encerrar_driver():
    """Fecha o navegador, se aberto. Chame ao final do crawl."""
    global _driver
    if _driver is not None:
        try:
            _driver.quit()
        except Exception:
            pass
        _driver = None
        print('[dinâmico] Chrome encerrado.')


def _pagina_ainda_em_desafio(driver):
    """Heurística rápida: o título/HTML atual ainda é a tela de desafio da Cloudflare?"""
    try:
        t = (driver.title or '').lower()
    except Exception:
        return False
    if any(sig in t for sig in ('just a moment', 'um momento', 'attention required')):
        return True
    try:
        corpo = driver.page_source[:3000].lower()
    except Exception:
        return False
    return 'verificação de segurança' in corpo or 'checking your browser' in corpo


def _texto_body(driver):
    """innerText atual do body (via JS, mais confiável que .text para SPAs)."""
    try:
        return driver.execute_script(
            "return document.body ? document.body.innerText : ''") or ""
    except Exception:
        return ""


def _esperar_conteudo_spa(driver):
    """Espera o conteúdo aparecer, em DUAS FASES independentes:

      Fase 1 — sair do desafio anti-bot (Cloudflare etc.), com backoff/reload.
      Fase 2 — surgir texto substancial (>= MIN_TEXTO_UTIL), rolando a página
               para disparar lazy-loading típico de SPAs.

    Devolve um status: 'ok' | 'bloqueio' | 'timeout_texto', além do tamanho do
    texto observado. Manter as fases separadas evita o problema antigo, em que
    uma condição composta (texto E sem-desafio) falhava silenciosamente e o
    HTML vazio era capturado como se fosse o conteúdo final.
    """
    import time as _time

    # Fase 1: desafio anti-bot
    for tentativa in range(1, DYN_MAX_TENTATIVAS + 1):
        if not _pagina_ainda_em_desafio(driver):
            break
        espera = 4 * tentativa  # 4s, 8s, 12s
        print(f'  [dinâmico] ainda no desafio anti-bot; aguardando {espera}s '
              f'(tentativa {tentativa}/{DYN_MAX_TENTATIVAS})...')
        _time.sleep(espera)
        try:
            driver.refresh()
        except Exception:
            pass
    if _pagina_ainda_em_desafio(driver):
        return 'bloqueio', len(_texto_body(driver).strip())

    # Fase 2: aguardar texto renderizado, rolando para acionar lazy-load
    fim = _time.time() + DYN_TIMEOUT_SPA
    ultimo = 0
    scrolls = 0
    while _time.time() < fim:
        ultimo = len(_texto_body(driver).strip())
        if ultimo >= MIN_TEXTO_UTIL:
            return 'ok', ultimo
        if scrolls < DYN_SCROLLS:
            try:
                driver.execute_script(
                    "window.scrollTo(0, document.body.scrollHeight);")
            except Exception:
                pass
            scrolls += 1
        _time.sleep(DYN_POLL_INTERVALO)
    return 'timeout_texto', ultimo


def baixar_pagina_dinamica(url):
    """Renderiza com Chrome e devolve o conteúdo já montado por JS.

    Fluxo: carrega a página, espera em duas fases (desafio anti-bot; depois
    conteúdo da SPA) e extrai o HTML final. Retorna o dict de sempre
    (origem='dinamico'), ou None se o driver estiver indisponível ou houver erro.

    Sinaliza o resultado da espera em dados['_status_dinamico'] para que
    coletar() possa registrar um motivo mais preciso no CSV de erros:
      - 'ok'            : conteúdo renderizado com sucesso
      - 'bloqueio'      : continuou na tela de desafio (tratado como bloqueio)
      - 'timeout_texto' : SPA não renderizou texto dentro de DYN_TIMEOUT_SPA
    """
    driver = _get_driver()
    if driver is None:
        return None
    try:
        driver.get(url)
        status, n_chars = _esperar_conteudo_spa(driver)

        if status == 'timeout_texto':
            print(f'  [dinâmico] SPA sem conteúdo renderizado após '
                  f'{DYN_TIMEOUT_SPA}s (texto={n_chars} chars) -> {url}')

        html = driver.page_source
        dados = _extrair_de_html(html, url)
        dados['origem'] = 'dinamico'
        dados['_status_dinamico'] = status
        return dados
    except Exception as e:
        print(f'  [ERRO dinâmico] {url}: {type(e).__name__}: {e}')
        return None


print('Motor de coleta dinâmica pronto (undetected-chromedriver + detecção de desafio).')


Motor de coleta dinâmica pronto (undetected-chromedriver + detecção de desafio).


## 4. O *crawler* (busca em largura) — com *fallback* dinâmico e registro de erros

Juntamos tudo numa **busca em largura (BFS)** com uma fila. Para cada página, o crawler:

1. tenta a **coleta estática** (`requests` + BeautifulSoup);
2. se o conteúdo central vier **vazio ou quase vazio** (texto abaixo de `MIN_TEXTO_UTIL`), aciona o
   **fallback dinâmico** (Chrome headless via Selenium), que executa o JavaScript e devolve o HTML
   renderizado para as mesmas funções de extração;
3. se mesmo assim não houver conteúdo aproveitável — ou se o navegador estiver indisponível — a URL é
   **registrada no CSV de erros** (`ARQUIVO_ERROS`) e o crawl **segue sem parar**.

Outros pontos que se mantêm: **links já filtrados** (só do conteúdo central, então o crawler não
persegue menus/rodapés), **profundidade** controlada por `PROFUNDIDADE_MAX`, e o teto `MAX_PAGINAS`
como salvaguarda. O navegador headless, quando necessário, é **aberto uma vez e reaproveitado**,
fechando ao final (mesmo se ocorrer erro no meio).

> **Reprocessar depois.** O `paginas_com_erro.csv` tem as colunas `url, profundidade, pai, motivo`.
> A última seção do notebook mostra como recarregá-lo como nova lista de sementes para uma segunda tentativa.


In [10]:
def coletar(paginas_semente, profundidade_max=2, mesmo_dominio_only=True,
            max_paginas=150, max_por_dominio=40, delay=0.5, respeitar_robots=True,
            arquivo_saida='corpus_coletado.json', intervalo_salvar=10,
            arquivo_erros='paginas_com_erro.csv',
            arquivo_ignoradas='paginas_ignoradas_por_limite.csv'):
    """Crawl em largura com salvamento incremental, retomada e fallback dinâmico.

    Fluxo por página:
      1. tenta a coleta ESTÁTICA (requests + BeautifulSoup);
      2. se o conteúdo vier vazio/quase vazio, tenta o FALLBACK DINÂMICO (Chrome headless);
      3. se ainda assim não houver conteúdo (ou o navegador estiver indisponível),
         registra a URL em `arquivo_erros` (CSV) e segue — sem interromper o crawl.

    O CSV de erros tem colunas: url, profundidade, pai, motivo. Você pode reusá-lo
    depois como nova lista de sementes para uma segunda tentativa.
    """
    import csv as _csv

    # arquivo local de trabalho (rápido); o destino final pode estar no Drive
    local = os.path.join('/content', '_crawl_local.json') if os.path.isdir('/content') \
            else os.path.basename(arquivo_saida)

    _pasta = os.path.dirname(arquivo_saida)
    if _pasta:
        os.makedirs(_pasta, exist_ok=True)

    # --- registro de páginas com erro (para reprocessar depois) ---
    erros = []          # lista de dicts: url, profundidade, pai, motivo
    erros_vistos = set()

    def _registrar_erro(url, prof, pai, motivo):
        if url in erros_vistos:
            return
        erros_vistos.add(url)
        erros.append({'url': url, 'profundidade': prof,
                      'pai': pai or '', 'motivo': motivo})
        try:
            with open(arquivo_erros, 'w', encoding='utf-8', newline='') as f:
                w = _csv.DictWriter(f, fieldnames=['url', 'profundidade', 'pai', 'motivo'])
                w.writeheader()
                w.writerows(erros)
        except OSError as e:
            print(f'  [AVISO] não consegui gravar {arquivo_erros}: {e}')

    # --- registro de páginas IGNORADAS por limite (cota de domínio ou teto global) ---
    # Estas URLs NÃO são falhas de coleta: elas eram válidas e estavam na fila, mas
    # foram puladas por causa de um limite (max_por_dominio ou max_paginas). Sem este
    # registro, elas 'sumiam' sem aparecer nem no JSON nem no CSV de erros — o que
    # torna difícil saber que existiram. Aqui ficam num CSV separado, para você poder
    # revisá-las (ex.: aumentar o limite e recoletar só o que ficou de fora).
    ignoradas = []      # dicts: url, profundidade, pai, motivo
    ignoradas_vistas = set()

    def _registrar_ignorada(url, prof, pai, motivo):
        if url in ignoradas_vistas:
            return
        ignoradas_vistas.add(url)
        ignoradas.append({'url': url, 'profundidade': prof,
                          'pai': pai or '', 'motivo': motivo})
        try:
            with open(arquivo_ignoradas, 'w', encoding='utf-8', newline='') as f:
                w = _csv.DictWriter(f, fieldnames=['url', 'profundidade', 'pai', 'motivo'])
                w.writeheader()
                w.writerows(ignoradas)
        except OSError as e:
            print(f'  [AVISO] não consegui gravar {arquivo_ignoradas}: {e}')

    # --- retomada: tenta o destino final; se não houver, tenta o local ---
    resultados = []
    visitadas = set()
    por_dominio = Counter()
    for origem in (arquivo_saida, local):
        try:
            with open(origem, 'r', encoding='utf-8') as f:
                resultados = json.load(f)
            for r in resultados:
                visitadas.add(r['url'])
                por_dominio[urlparse(r['url']).netloc.lower()] += 1
            print(f'[RETOMADA] {len(resultados)} página(s) já coletadas em {origem} — não serão refeitas.')
            break
        except (FileNotFoundError, json.JSONDecodeError):
            continue

    def _gravar(caminho):
        try:
            with open(caminho, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=2)
            return True
        except OSError as e:
            print(f'  [AVISO] não consegui gravar {caminho}: {e}')
            return False

    def _sincronizar():
        if local != arquivo_saida:
            try:
                shutil.copyfile(local, arquivo_saida)
            except OSError as e:
                print(f'  [AVISO] não consegui sincronizar com {arquivo_saida}: {e}')

    fila = deque()
    for url in paginas_semente:
        u = normalizar_url(url, url)
        if u:
            fila.append((u, 0, None))

    if resultados:
        for r in resultados:
            prof_r = r.get('profundidade', 0)
            if profundidade_max is None or prof_r < profundidade_max:
                for link in r.get('links', []):
                    if mesmo_dominio_only and not mesmo_dominio(r['url'], link):
                        continue
                    if link not in visitadas:
                        fila.append((link, prof_r + 1, r['url']))

    desde_ultimo_sync = 0
    try:
        while fila:
            if len(resultados) >= max_paginas:
                print(f'\n[LIMITE] teto global de {max_paginas} páginas atingido.')
                # O que ainda restou na fila não será coletado: registra para
                # transparência. (A verificação ocorre ANTES do popleft, então
                # nada foi retirado ainda; a fila contém exatamente o que falta.)
                for _u, _p, _pai in fila:
                    _registrar_ignorada(_u, _p, _pai, f'teto_global ({max_paginas} paginas)')
                break
            url, prof, pai = fila.popleft()
            if url in visitadas:
                continue
            visitadas.add(url)
            dom = urlparse(url).netloc.lower()
            if max_por_dominio is not None and por_dominio[dom] >= max_por_dominio:
                _registrar_ignorada(url, prof, pai,
                                    f'limite_por_dominio ({max_por_dominio} de {dom})')
                continue
            if not pode_acessar(url, respeitar_robots):
                print(f'[BLOQUEADO robots.txt] {url}')
                _registrar_erro(url, prof, pai, 'bloqueado_robots')
                continue
            print(f'[{len(resultados)+1}/{max_paginas}] nível {prof} | fila: {len(fila)} | {url}')

            # 1) tentativa estática
            dados = baixar_pagina(url)
            time.sleep(delay)

            # Decide se precisa do fallback: conteúdo vazio OU página de bloqueio anti-bot.
            # Mas nunca para arquivos que claramente não são HTML (.pdf, .rtf, .zip...).
            precisa_dinamico = (_conteudo_vazio(dados) or _parece_bloqueio(dados)) \
                               and not _extensao_nao_html(url)

            # 2) fallback dinâmico
            if precisa_dinamico:
                if dados is None:
                    motivo_estatico = 'estatico_falhou'
                elif _parece_bloqueio(dados):
                    motivo_estatico = 'estatico_bloqueado'
                else:
                    motivo_estatico = 'estatico_vazio'
                print(f'  [fallback] estático insuficiente ({motivo_estatico}) -> renderização dinâmica...')
                din = baixar_pagina_dinamica(url)

                # Avalia o resultado do dinâmico, nesta ordem:
                if _parece_bloqueio(din):
                    # O navegador recebeu a página de desafio (Cloudflare etc.) — NÃO salvar.
                    _registrar_erro(url, prof, pai, 'bloqueio_anti_bot')
                    print(f'  [erro registrado] {url} -> bloqueio_anti_bot (desafio não resolvido)')
                    continue
                elif not _conteudo_vazio(din):
                    din.pop('_status_dinamico', None)  # não vai para o JSON final
                    dados = din
                else:
                    # Motivo mais preciso quando o dinâmico rodou mas a SPA não
                    # renderizou texto a tempo (distinto de 'dinâmico indisponível').
                    if din is None:
                        motivo = f'{motivo_estatico}_e_dinamico_indisponivel'
                    elif din.get('_status_dinamico') == 'timeout_texto':
                        motivo = 'spa_sem_conteudo_renderizado'
                    else:
                        motivo = 'sem_conteudo_estatico_e_dinamico'
                    _registrar_erro(url, prof, pai, motivo)
                    print(f'  [erro registrado] {url} -> {motivo}')
                    continue

            # Salvaguarda: se chegou aqui sem conteúdo aproveitável (ex.: arquivo
            # não-HTML como .rtf, cujo estático retornou None e que o guarda de
            # extensão impediu de ir ao dinâmico), registra erro e segue — nunca
            # tenta indexar um dados == None no append abaixo.
            if _conteudo_vazio(dados):
                motivo = 'nao_html_ou_sem_conteudo' if _extensao_nao_html(url) \
                         else 'sem_conteudo'
                _registrar_erro(url, prof, pai, motivo)
                print(f'  [erro registrado] {url} -> {motivo}')
                continue

            # Salvaguarda final: mesmo sem passar pelo fallback, nunca salvar um bloqueio.
            if _parece_bloqueio(dados):
                _registrar_erro(url, prof, pai, 'bloqueio_anti_bot')
                print(f'  [erro registrado] {url} -> bloqueio_anti_bot')
                continue

            por_dominio[dom] += 1
            resultados.append({
                'url': url, 'titulo': dados['titulo'], 'texto': dados['texto'],
                'relacao': 'raiz' if pai is None else 'filho', 'pai': pai,
                'profundidade': prof, 'links': dados['links'], 'figuras': dados['figuras'],
                'origem': dados.get('origem', 'estatico'),
            })
            _gravar(local)
            desde_ultimo_sync += 1
            if desde_ultimo_sync >= intervalo_salvar:
                _sincronizar()
                desde_ultimo_sync = 0
                print(f'  [sync] {len(resultados)} páginas salvas em {arquivo_saida}')
            if profundidade_max is None or prof < profundidade_max:
                for link in dados['links']:
                    if mesmo_dominio_only and not mesmo_dominio(url, link):
                        continue
                    if link not in visitadas:
                        fila.append((link, prof + 1, url))
    finally:
        # encerra o navegador (se foi aberto) mesmo se algo der errado
        try:
            encerrar_driver()
        except Exception:
            pass

    _gravar(local)
    _sincronizar()
    print(f'\n[FIM] {len(resultados)} página(s) salvas em {arquivo_saida}')
    if erros:
        print(f'[ERROS] {len(erros)} página(s) registradas em {arquivo_erros} para reprocessar.')
    if ignoradas:
        print(f'[IGNORADAS] {len(ignoradas)} página(s) puladas por limite '
              f'(cota de domínio / teto global) registradas em {arquivo_ignoradas}.')
        print(f'            Para coletá-las, aumente max_por_dominio / max_paginas '
              f'e use este CSV como sementes (profundidade_max=0).')
    return resultados


## 5. Executando a coleta

Agora é só chamar a função. Com profundidade ilimitada, isso pode levar bastante tempo em sites grandes
(lembre-se da pausa entre páginas e do teto `MAX_PAGINAS`). Acompanhe o log para ver o avanço por nível.

In [11]:
resultados = coletar(
    PAGINAS_SEMENTE,
    profundidade_max=PROFUNDIDADE_MAX,
    mesmo_dominio_only=MESMO_DOMINIO,
    max_paginas=MAX_PAGINAS,
    max_por_dominio=MAX_POR_DOMINIO,
    delay=DELAY_SEGUNDOS,
    arquivo_saida=ARQUIVO_SAIDA,     # mesmo nome usado na configuração
    intervalo_salvar=INTERVALO_SALVAR,
    arquivo_erros=ARQUIVO_ERROS,     # CSV com as páginas que falharam (p/ reprocessar)
    arquivo_ignoradas=ARQUIVO_IGNORADAS,  # CSV com páginas puladas por limite (transparência)
)

print(f'\nTotal de páginas coletadas: {len(resultados)}')
raizes = sum(1 for r in resultados if r['relacao'] == 'raiz')
filhos = sum(1 for r in resultados if r['relacao'] == 'filho')
total_figuras = sum(len(r['figuras']) for r in resultados)
din = sum(1 for r in resultados if r.get('origem') == 'dinamico')
prof_max_vista = max((r['profundidade'] for r in resultados), default=0)
print(f'  Raízes: {raizes} | Filhos: {filhos}')
print(f'  Coletadas via renderização dinâmica (JS): {din}')
print(f'  Profundidade máxima alcançada: {prof_max_vista}')
print(f'  Total de figuras coletadas: {total_figuras}')


[1/3000] nível 0 | fila: 47 | https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689
  [fallback] estático insuficiente (estatico_vazio) -> renderização dinâmica...
[dinâmico] undetected-chromedriver indisponível (ModuleNotFoundError). Tentando Selenium comum...
[dinâmico] Selenium Chrome comum iniciado.
[2/3000] nível 0 | fila: 48 | https://www.iso.org/standard/56641.html
[3/3000] nível 0 | fila: 53 | https://www.iso.org/standard/74296.html
[4/3000] nível 0 | fila: 61 | https://www.iso.org/standard/78507.html
[5/3000] nível 0 | fila: 65 | https://www.iso.org/standard/42001
[6/3000] nível 0 | fila: 81 | https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13709.htm
[7/3000] nível 0 | fila: 106 | https://oecd.ai/en/ai-principles
[8/3000] nível 0 | fila: 106 | https://www.gov.br/anpd/pt-br/acesso-a-informacao/institucional/atos-normativos/regulamentacoes_anpd/resolucao-cd-anpd-no-2-de-27-de-janeiro-de-2022
[9/3000] nível 0 | fila: 105 | https://ico.org.uk/for-organ

## 6. Conferindo o resultado

Antes de salvar, vamos espiar os primeiros registros para verificar se os campos foram preenchidos
corretamente (mostramos o texto truncado e listamos as primeiras figuras).

In [12]:
for r in resultados[:3]:
    print('URL        :', r['url'])
    print('Título     :', r['titulo'])
    print('Relação    :', r['relacao'], '| Pai:', r['pai'], '| Nível:', r['profundidade'])
    print('Nº links   :', len(r['links']))
    print('Nº figuras :', len(r['figuras']))
    for fig in r['figuras'][:3]:
        rotulo = fig['alt'] or fig['legenda'] or '(sem descrição)'
        print(f'    - [{fig["tipo"]}] {fig["url"]}  «{rotulo}»')
    print('Texto      :', r['texto'][:500], '...')
    print('-' * 80)

URL        : https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689
Título     : Regulation - EU - 2024/1689 - EN - EUR-Lex
Relação    : raiz | Pai: None | Nível: 0
Nº links   : 3
Nº figuras : 3
    - [imagem] https://webtools.europa.eu/images/flags/eu.svg  «(sem descrição)»
    - [svg] None  «(sem descrição)»
    - [svg] None  «(sem descrição)»
Texto      : An official website of the European Union An official EU website How do you know? This document is an excerpt from the EUR-Lex website Search tips Need more search options? Use the Advanced search Document 32024R1689 Help Print Share ...
--------------------------------------------------------------------------------
URL        : https://www.iso.org/standard/56641.html
Título     : ISO/IEC 38507:2022 - Information technology — Governance of IT — Governance implications of the use of artificial intelligence by organizations
Relação    : raiz | Pai: None | Nível: 0
Nº links   : 8
Nº figuras : 3
    - [imagem] https://ww

## 7. Salvando em JSON

Gravamos a lista de registros no arquivo definido em `ARQUIVO_SAIDA`. Usamos `ensure_ascii=False` para
preservar os acentos e `indent=2` para deixar o arquivo legível.

In [13]:
with open(ARQUIVO_SAIDA, 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print(f'Conteúdo salvo em: {ARQUIVO_SAIDA}')
print(f'{len(resultados)} registro(s) gravado(s).')

# No Google Colab, descomente para baixar o arquivo:
# from google.colab import files
# files.download(ARQUIVO_SAIDA)

Conteúdo salvo em: conteudo_coletado.json
404 registro(s) gravado(s).


## 8. Relendo o JSON para conferir

Por fim, relemos o arquivo salvo e mostramos o primeiro registro formatado — uma forma simples de
garantir que o JSON foi gravado corretamente e está pronto para uso em outras etapas (análise, busca,
indexação etc.).

In [14]:
with open(ARQUIVO_SAIDA, 'r', encoding='utf-8') as f:
    dados_lidos = json.load(f)

print(f'O arquivo contém {len(dados_lidos)} registro(s).\n')
print('Estrutura do primeiro registro (texto, links e figuras truncados):')

exemplo = dict(dados_lidos[0])
exemplo['texto'] = exemplo['texto'][:150] + ' ...'
exemplo['links'] = exemplo['links'][:5] + (['...'] if len(exemplo['links']) > 5 else [])
exemplo['figuras'] = exemplo['figuras'][:3] + (['...'] if len(exemplo['figuras']) > 3 else [])
print(json.dumps(exemplo, ensure_ascii=False, indent=2))

O arquivo contém 404 registro(s).

Estrutura do primeiro registro (texto, links e figuras truncados):
{
  "url": "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689",
  "titulo": "Regulation - EU - 2024/1689 - EN - EUR-Lex",
  "texto": "An official website of the European Union An official EU website How do you know? This document is an excerpt from the EUR-Lex website Search tips Nee ...",
  "relacao": "raiz",
  "pai": null,
  "profundidade": 0,
  "links": [
    "https://eur-lex.europa.eu/advanced-search-form.html",
    "https://eur-lex.europa.eu/content/help.html",
    "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689"
  ],
  "figuras": [
    {
      "url": "https://webtools.europa.eu/images/flags/eu.svg",
      "alt": "",
      "legenda": "",
      "tipo": "imagem"
    },
    {
      "url": null,
      "alt": "",
      "legenda": "",
      "tipo": "svg"
    },
    {
      "url": null,
      "alt": "",
      "legenda": "",
      "tipo": "svg"
   

## 8.5 (Opcional) Limpar um JSON já coletado que contém páginas de bloqueio

Se você já rodou o crawl **antes** desta correção, o `conteudo_coletado.json` pode conter páginas de
desafio anti-bot salvas por engano (ex.: as da ISO, com título *"Um momento…"*). O bloco abaixo
**remove** essas entradas do JSON e as acrescenta ao `paginas_com_erro.csv`, para você reprocessá-las
depois — sem precisar refazer toda a coleta.


In [15]:
import json, csv, os

def limpar_bloqueios_do_json(caminho_json=ARQUIVO_SAIDA, caminho_erros=ARQUIVO_ERROS):
    """Remove páginas de bloqueio anti-bot de um JSON já coletado e as manda p/ o CSV de erros."""
    if not os.path.exists(caminho_json):
        print(f'{caminho_json} não encontrado.'); return
    with open(caminho_json, 'r', encoding='utf-8') as f:
        dados = json.load(f)

    limpos, bloqueados = [], []
    for r in dados:
        if _parece_bloqueio(r):
            bloqueados.append(r)
        else:
            limpos.append(r)

    if not bloqueados:
        print('Nenhuma página de bloqueio encontrada no JSON. Nada a fazer.'); return

    # Reescreve o JSON sem os bloqueios
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(limpos, f, ensure_ascii=False, indent=2)

    # Acrescenta os bloqueios ao CSV de erros (sem duplicar URLs já lá)
    ja_tem = set()
    if os.path.exists(caminho_erros):
        with open(caminho_erros, 'r', encoding='utf-8') as f:
            ja_tem = {l['url'] for l in csv.DictReader(f)}
    novo = os.path.exists(caminho_erros)
    with open(caminho_erros, 'a', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['url','profundidade','pai','motivo'])
        if not novo:
            w.writeheader()
        for r in bloqueados:
            if r['url'] in ja_tem:
                continue
            w.writerow({'url':r['url'], 'profundidade':r.get('profundidade',0),
                        'pai':r.get('pai') or '', 'motivo':'bloqueio_anti_bot'})

    print(f'Removidas {len(bloqueados)} página(s) de bloqueio do JSON '
          f'({len(limpos)} mantidas). URLs adicionadas a {caminho_erros}:')
    for r in bloqueados:
        print('  -', r['url'])


# Descomente para executar sobre o seu arquivo já coletado:
# limpar_bloqueios_do_json()


## 9. (Opcional) Reprocessar as páginas que falharam

Se o `paginas_com_erro.csv` tiver entradas (por exemplo, SPAs que agora a espera melhorada
consegue renderizar, ou páginas que falharam porque o Selenium não estava instalado antes),
você pode recarregá-lo e coletar **apenas essas URLs**, **anexando ao JSON existente**.

Dois pontos garantem que nada é refeito nem perdido:

- **Filtro de recuperáveis:** por padrão, URLs que apontam para arquivos **não-HTML**
  (`.pdf`, `.docx`, `.doc`, `.zip`, ...) são **puladas** — elas falhariam de novo pelo mesmo
  motivo. Passe `apenas_recuperaveis=False` se quiser tentar todas.
- **Retomada / anexar:** o `coletar()` lê o JSON já existente, **não refaz** as páginas já
  coletadas e **acrescenta** as novas ao mesmo arquivo.

Para executar, ajuste `REPROCESSAR = True` na célula abaixo (com `profundidade_max=0`, ele
reprocessa só as próprias páginas do CSV, sem seguir novos links).


In [16]:
import csv as _csv, os

# Motivos que NÃO vale a pena reprocessar: são arquivos que não são HTML
# (.pdf/.docx/...), que falhariam de novo pelo mesmo motivo. O filtro principal
# é por EXTENSÃO (mesmo critério do crawler: _extensao_nao_html), o que já cobre
# as linhas com motivo 'nao_html_ou_sem_conteudo'.
def carregar_sementes_de_erros(caminho=ARQUIVO_ERROS, apenas_recuperaveis=True):
    """Lê o CSV de erros e devolve as URLs para uma nova tentativa.

    Se apenas_recuperaveis=True (padrão), pula URLs que apontam para arquivos
    não-HTML (.pdf, .docx, .doc, .zip, ...), que não têm como ser coletadas como
    página e só voltariam para o CSV de erros.
    """
    if not os.path.exists(caminho):
        print(f'Nenhum arquivo de erros em {caminho} — nada a reprocessar.')
        return []
    with open(caminho, 'r', encoding='utf-8') as f:
        linhas = list(_csv.DictReader(f))

    urls, pulados = [], []
    for l in linhas:
        u = l.get('url')
        if not u:
            continue
        if apenas_recuperaveis and _extensao_nao_html(u):
            pulados.append(l)
        else:
            urls.append(u)

    print(f'{len(linhas)} linha(s) no CSV de erros.')
    print(f'  -> {len(urls)} recuperável(is) (HTML/SPA) para reprocessar:')
    for u in urls[:20]:
        print(f'       {u}')
    if pulados:
        print(f'  -> {len(pulados)} pulada(s) por serem arquivos não-HTML '
              f'(.pdf/.docx/...), que falhariam de novo:')
        for l in pulados[:20]:
            print(f'       [{l.get("motivo","?")}] {l["url"]}')
    return urls


sementes_erro = carregar_sementes_de_erros(apenas_recuperaveis=True)

# ---------------------------------------------------------------------------
# Reprocessamento: coleta SÓ as páginas que falharam e ANEXA ao JSON existente.
#
# Como funciona o "anexar": coletar() faz RETOMADA automática — ao iniciar, lê o
# ARQUIVO_SAIDA já existente, marca as URLs já coletadas como visitadas (não as
# refaz) e acrescenta as novas ao MESMO arquivo. Ou seja, o JSON existente é
# preservado e apenas complementado.
#
# profundidade_max=0  -> reprocessa apenas as próprias URLs do CSV, sem seguir
#                        novos links (não recria o crawl inteiro).
#
# Deixe REPROCESSAR = True para executar de fato.
# ---------------------------------------------------------------------------
REPROCESSAR = False   # mude para True quando quiser rodar

if REPROCESSAR and sementes_erro:
    resultados = coletar(
        sementes_erro,
        profundidade_max=0,                      # só as páginas que falharam
        mesmo_dominio_only=MESMO_DOMINIO,
        max_paginas=MAX_PAGINAS,
        max_por_dominio=MAX_POR_DOMINIO,
        delay=DELAY_SEGUNDOS,
        arquivo_saida=ARQUIVO_SAIDA,             # ANEXA ao mesmo JSON (via retomada)
        intervalo_salvar=INTERVALO_SALVAR,
        arquivo_erros='paginas_com_erro_2.csv',  # o que ainda falhar vai para um CSV novo
    )
    encerrar_driver()                            # fecha o Chrome do fallback dinâmico
    print(f'\nReprocessamento concluído: {len(resultados)} página(s) no JSON agora.')
elif not REPROCESSAR:
    print('\n[info] REPROCESSAR=False — nada foi coletado. '
          'Mude para True na célula para executar.')


20 linha(s) no CSV de erros.
  -> 9 recuperável(is) (HTML/SPA) para reprocessar:
       https://bvsms.saude.gov.br/bvs/saudelegis/gm/2020/prt3632_22_12_2020.html
       https://bvsms.saude.gov.br/bvs/saudelegis/cns/2013/res0466_12_12_2012.html
       https://bvsms.saude.gov.br/bvs/saudelegis/cns/2016/res0510_07_04_2016.html
       https://bvsms.saude.gov.br/bvs/saudelegis/gm/2017/prt0271_27_01_2017.html
       https://bvsms.saude.gov.br/bvs/saudelegis/gm/2021/prt1768_02_08_2021.html
       https://bvsms.saude.gov.br/bvs/saudelegis/cns/2022/res0659_15_06_2022.html
       https://meususdigital.saude.gov.br/publico/perfil/sobre-sus
       https://www.planalto.gov.br/ccivil_03/Constituicao/Constituiçao.htm
       http://www.ufmg.br/bioetica/coep
  -> 11 pulada(s) por serem arquivos não-HTML (.pdf/.docx/...), que falhariam de novo:
       [nao_html_ou_sem_conteudo] https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/Ret/LEI13709-Repart52.rtf
       [nao_html_ou_sem_conteudo] https://ic

## Observações finais

- **Profundidade**: `PROFUNDIDADE_MAX = None` percorre **todos os níveis** alcançáveis no domínio.
  Para limitar, defina um número (ex.: `2`). O teto `MAX_PAGINAS` evita crawls grandes demais.
- **Conteúdo central**: a filtragem de menus/rodapés/sidebars é heurística. Se um site usar nomes de
  `class`/`id` incomuns, ajuste as listas `_LIXO_PADROES` e `_SELETORES_PRINCIPAIS` na seção 3.3.
- **Figuras**: imagens são coletadas com URL absoluta; diagramas em `<svg>` inline aparecem com
  `url = null` (são vetoriais, sem arquivo próprio) e o rótulo possível em `alt`.
- **Outros domínios**: defina `MESMO_DOMINIO = False` para seguir links externos também (cuidado: o
  crawl pode crescer sem controle).
- **Educação e ética**: mantenha o `DELAY_SEGUNDOS`, respeite o `robots.txt` e verifique os termos de
  uso do site antes de coletar dados em escala.


## 10. Coleta estruturada de normas ISO via `obp-access`

As URLs da plataforma **OBP da ISO** ficam atrás do desafio anti-bot da Cloudflare
e caem no `paginas_com_erro.csv` com motivo `bloqueio_anti_bot` (comportamento
correto: não salvamos a página de desafio). Esta seção tenta recuperar **apenas o
conteúdo informativo e gratuito** dessas normas (prefácio, introdução, escopo,
termos e definições, referências) usando o projeto **`obp-access`** da Metanorma,
que fala com o backend oficial do OBP e devolve o conteúdo em **XML no formato
NISO STS**. Convertemos esse XML em **texto corrido**, no mesmo formato dos demais
registros do JSON.

### ⚠️ Limitações importantes (leia antes de usar)

1. **`obp-access` é uma gem Ruby**, não uma biblioteca Python. Para usá-la aqui
   você precisa de **Ruby (2.7+) e a gem instalada** no ambiente. Instale conforme
   onde você roda o notebook:

   **Google Colab / Ubuntu / Debian:**
   ```bash
   !apt-get -qq install -y ruby-full >/dev/null && sudo gem install obp-access
   ```

   **Linux (Fedora/RHEL):**
   ```bash
   sudo dnf install -y ruby ruby-devel && gem install obp-access
   ```

   **macOS (Homebrew):**
   ```bash
   brew install ruby        # o Ruby do sistema é antigo; instale um atual
   gem install obp-access   # pode precisar de: sudo gem install obp-access
   ```

   **Windows:** instale o **RubyInstaller** (https://rubyinstaller.org/) —
   marque a opção *Add Ruby to PATH* durante a instalação — e depois, no
   prompt/PowerShell:
   ```bash
   gem install obp-access
   ```

   Para conferir se ficou tudo certo (em qualquer sistema), rode no terminal:
   ```bash
   ruby --version        # deve mostrar 2.7 ou superior
   obp-access --help     # deve encontrar o comando
   ```
   Se a ferramenta não estiver instalada, a etapa **avisa e é ignorada** — nada
   quebra, e as URLs ISO continuam no CSV de erros.

2. **Não é uma garantia de contornar a Cloudflare.** O `obp-access` acessa o
   mesmo backend OBP; ele apenas estrutura melhor o que a ISO libera de graça.
   Se o acesso falhar, a norma **volta ao CSV** com motivo `obp_access_falhou`.
   Não adicionamos aqui rotação de proxies, resolução de CAPTCHA, nem qualquer
   técnica para burlar a proteção — o objetivo é só a **prévia pública** que a
   própria ISO oferece.

3. **Só coletamos a prévia pública gratuita**, nunca o texto normativo pago.

4. URLs ISO no formato `iso.org/standard/NNNN.html` **não** carregam o
   identificador (URN) da norma no endereço; para essas, não há como montar a
   consulta e elas são marcadas como `iso_sem_urn_no_fragmento`. Use as URLs do
   OBP (`iso.org/obp/ui/...#iso:std:...`) para que a coleta funcione.

A etapa é **opt-in** e **separada do crawl**: ela lê o `paginas_com_erro.csv`,
processa só as linhas da ISO, e anexa os resultados ao JsON — sem refazer nada do
que já foi coletado.

> **Ordem recomendada de execução:** esta seção **10 (ISO)** roda *antes* da
> seção **11 (pós-processamento)**, para que as normas ISO recém-coletadas também
> passem pela limpeza de vazios/lixo antes de irem para o RAG. A seção 11 é idempotente: se precisar, pode rodá-la de novo depois desta sem problemas.

In [17]:
"""
Coleta estruturada de normas ISO via `obp-access` (gem Ruby da Metanorma).

Estratégia:
  - Detecta URLs de norma ISO no CSV de erros (motivo 'bloqueio_anti_bot').
  - Extrai o URN da norma (ex.: iso:std:iso-iec:38507:ed-1:v1:en).
  - Chama a CLI `obp-access fetch <urn>`, que devolve XML no formato NISO STS
    (só o conteúdo INFORMATIVO e gratuito: prefácio, escopo, termos, referências).
  - Faz o parse do XML e monta um campo `texto` corrido, compatível com os demais
    registros do JSON.

IMPORTANTE (limites):
  - Requer Ruby + a gem `obp-access` instalados no ambiente (veja instruções na
    célula do notebook). Se não estiverem, cada URN é registrada como erro e a
    etapa segue sem quebrar.
  - O `obp-access` acessa o mesmo backend OBP da ISO. NÃO é garantia de contornar
    a Cloudflare; é apenas um cliente que estrutura melhor o que a ISO libera
    gratuitamente. Se o acesso falhar, a URN vai para o CSV de erros.
  - Coletamos apenas o material que a própria ISO disponibiliza como prévia
    pública. Nada de conteúdo normativo pago.
"""
import json, re, os, csv, subprocess, shutil
import xml.etree.ElementTree as ET
from urllib.parse import urlparse, unquote


# --- Detecção / extração do identificador da norma -------------------------------
def eh_url_iso(url):
    return 'iso.org' in urlparse(url).netloc.lower()


def extrair_urn_iso(url):
    """Extrai o URN da norma (ex.: 'iso:std:iso-iec:38507:ed-1:v1:en') da URL do OBP.

    Só funciona para URLs do OBP com o URN no fragmento (#iso:std:...).
    URLs do tipo /standard/56641.html NÃO carregam o URN e retornam None.
    """
    frag = unquote(urlparse(url).fragment or '')
    m = re.search(r'(iso:std:[a-z0-9:\-]+)', frag, re.IGNORECASE)
    return m.group(1) if m else None


# --- Parse do XML NISO STS -> texto corrido --------------------------------------
def _local(tag):
    """Remove namespace de um tag ElementTree ('{ns}sec' -> 'sec')."""
    return tag.split('}', 1)[-1] if '}' in tag else tag


def _texto_de(elem):
    """Texto corrido de um elemento e todos os descendentes, normalizando espaços."""
    partes = []
    for t in elem.itertext():
        if t and t.strip():
            partes.append(t.strip())
    return ' '.join(' '.join(partes).split())


def _titulo_sec(sec):
    for filho in sec:
        if _local(filho.tag) == 'title':
            return _texto_de(filho)
    return ''


def parse_sts_para_texto(xml_str):
    """Converte XML NISO STS em (titulo, texto_corrido).

    Percorre <front>/<body>/<back>, e para cada <sec> concatena o título e o
    texto. Também trata pares <term>/<def> (glossários) de forma legível.
    Genérico: não depende de conhecer todos os sec-type de antemão.
    """
    # remove declaração de encoding que às vezes atrapalha o ET.fromstring
    xml_str = re.sub(r'<\?xml[^>]*\?>', '', xml_str, count=1).strip()
    try:
        root = ET.fromstring(xml_str)
    except ET.ParseError as e:
        raise ValueError(f'XML STS inválido: {e}')

    # título: <std-ref> ou <title> em iso-meta, com fallback
    titulo = ''
    for elem in root.iter():
        if _local(elem.tag) in ('std-ref', 'doc-ref') and _texto_de(elem):
            titulo = _texto_de(elem)
            break
    if not titulo:
        for elem in root.iter():
            if _local(elem.tag) == 'title' and _texto_de(elem):
                titulo = _texto_de(elem)
                break
    titulo = titulo or '(norma ISO sem título)'

    blocos = []

    def processar_sec(sec, nivel=0):
        titulo_sec = _titulo_sec(sec)
        if titulo_sec:
            blocos.append(('#' * (nivel + 1)) + ' ' + titulo_sec)
        for filho in sec:
            nome = _local(filho.tag)
            if nome == 'title':
                continue
            elif nome == 'label':
                # numeração da seção (ex.: "3", "4.1") — ruído para o RAG
                continue
            elif nome == 'sec':
                processar_sec(filho, nivel + 1)
            elif nome in ('term-sec',):
                # glossário estruturado: term-display -> term / def
                termo = ''
                definicao = ''
                for e in filho.iter():
                    ln = _local(e.tag)
                    if ln == 'term' and not termo:
                        termo = _texto_de(e)
                    elif ln == 'def' and not definicao:
                        definicao = _texto_de(e)
                if termo or definicao:
                    blocos.append(f'{termo}: {definicao}'.strip(': ').strip())
            elif nome in ('def-list', 'glossary'):
                for di in filho.iter():
                    if _local(di.tag) == 'def-item':
                        termo = definicao = ''
                        for e in di:
                            if _local(e.tag) == 'term':
                                termo = _texto_de(e)
                            elif _local(e.tag) == 'def':
                                definicao = _texto_de(e)
                        if termo or definicao:
                            blocos.append(f'{termo}: {definicao}'.strip(': ').strip())
            else:
                txt = _texto_de(filho)
                if txt:
                    blocos.append(txt)

    # front (prefácio/introdução), body (escopo, termos...), back (referências)
    for parte in root.iter():
        if _local(parte.tag) in ('front', 'body', 'back'):
            for filho in parte:
                if _local(filho.tag) == 'sec':
                    processar_sec(filho)

    texto = '\n'.join(b for b in blocos if b.strip())
    texto = re.sub(r'\n{3,}', '\n\n', texto).strip()
    return titulo, texto


# --- Chamada à CLI do obp-access -------------------------------------------------
def obp_access_disponivel():
    return shutil.which('obp-access') is not None


def buscar_sts_via_obp(urn, timeout=60):
    """Chama `obp-access fetch <urn>` e devolve o XML STS (str) ou levanta erro."""
    if not obp_access_disponivel():
        raise RuntimeError('obp-access não encontrado no PATH. '
                           'Instale com: gem install obp-access')
    proc = subprocess.run(
        ['obp-access', 'fetch', urn],
        capture_output=True, text=True, timeout=timeout,
    )
    if proc.returncode != 0:
        raise RuntimeError(f'obp-access falhou (rc={proc.returncode}): '
                           f'{proc.stderr.strip()[:300]}')
    if not proc.stdout.strip():
        raise RuntimeError('obp-access retornou saída vazia.')
    return proc.stdout


# --- Orquestração: lê CSV de erros, processa ISO, anexa ao JSON ------------------
def processar_normas_iso(caminho_json, caminho_erros,
                         caminho_json_saida=None, timeout=60):
    """Reprocessa URLs ISO do CSV de erros via obp-access e anexa ao JSON.

    - Só toca em linhas cujo URL é da ISO.
    - Cada norma coletada com sucesso vira um registro novo no JSON, no MESMO
      formato dos demais (url, titulo, texto, relacao, pai, profundidade, links,
      figuras, origem='obp-access').
    - Falhas continuam no CSV de erros (motivo atualizado).
    """
    caminho_json_saida = caminho_json_saida or caminho_json

    with open(caminho_json, 'r', encoding='utf-8') as f:
        registros = json.load(f)
    urls_ja = {r['url'] for r in registros}

    if not os.path.exists(caminho_erros):
        print(f'{caminho_erros} não encontrado — nada a reprocessar.')
        return registros
    with open(caminho_erros, 'r', encoding='utf-8') as f:
        linhas_erro = list(csv.DictReader(f))

    iso_linhas = [l for l in linhas_erro if eh_url_iso(l.get('url', ''))]
    print(f'{len(iso_linhas)} URL(s) ISO no CSV de erros.')

    if not obp_access_disponivel():
        print('[AVISO] obp-access não está instalado — pulando a coleta ISO. '
              'Instale Ruby + `gem install obp-access` e rode esta célula de novo.')
        return registros

    novos, ainda_erro = 0, []
    for l in iso_linhas:
        url = l['url']
        if url in urls_ja:
            continue
        urn = extrair_urn_iso(url)
        if not urn:
            print(f'  [pulado] sem URN no fragmento: {url}')
            ainda_erro.append({**l, 'motivo': 'iso_sem_urn_no_fragmento'})
            continue
        try:
            xml = buscar_sts_via_obp(urn, timeout=timeout)
            titulo, texto = parse_sts_para_texto(xml)
            if len(texto.strip()) < 100:
                raise ValueError('conteúdo informativo muito curto/ausente')
            registros.append({
                'url': url, 'titulo': titulo, 'texto': texto,
                'relacao': l.get('pai') and 'filho' or 'raiz',
                'pai': l.get('pai') or None,
                'profundidade': int(l.get('profundidade') or 0),
                'links': [], 'figuras': [], 'origem': 'obp-access',
            })
            urls_ja.add(url)
            novos += 1
            print(f'  [ok] {urn} -> {len(texto)} caracteres')
        except Exception as e:
            print(f'  [falha] {urn or url}: {type(e).__name__}: {str(e)[:200]}')
            ainda_erro.append({**l, 'motivo': f'obp_access_falhou'})

    with open(caminho_json_saida, 'w', encoding='utf-8') as f:
        json.dump(registros, f, ensure_ascii=False, indent=2)

    # reescreve o CSV: mantém não-ISO como estavam + ISO que ainda falharam
    nao_iso = [l for l in linhas_erro if not eh_url_iso(l.get('url', ''))]
    with open(caminho_erros, 'w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['url', 'profundidade', 'pai', 'motivo'])
        w.writeheader()
        for l in nao_iso + ainda_erro:
            w.writerow({k: l.get(k, '') for k in ['url', 'profundidade', 'pai', 'motivo']})

    print(f'\n{novos} norma(s) ISO adicionada(s) ao JSON.')
    print(f'{len(ainda_erro)} ISO ainda em erro (registradas em {caminho_erros}).')
    return registros


In [18]:
# Reprocessa as URLs ISO do CSV de erros via obp-access e anexa ao JSON.
#
# Pré-requisito: Ruby (2.7+) + a gem obp-access no PATH. Descomente conforme o SO:
#   Colab/Ubuntu/Debian: !apt-get -qq install -y ruby-full >/dev/null && sudo gem install obp-access
#   Fedora/RHEL:         !sudo dnf install -y ruby ruby-devel && gem install obp-access
#   macOS (Homebrew):    !brew install ruby && gem install obp-access
#   Windows:             instale o RubyInstaller (rubyinstaller.org, marque "Add Ruby to PATH")
#                        e rode:  gem install obp-access
# Confira com:  !ruby --version   e   !obp-access --help
#
# Roda sobre o ARQUIVO_SAIDA (JSON coletado) e o ARQUIVO_ERROS (CSV) da config.

registros_atualizados = processar_normas_iso(
    caminho_json   = ARQUIVO_SAIDA,
    caminho_erros  = ARQUIVO_ERROS,
    caminho_json_saida = ARQUIVO_SAIDA,   # grava no próprio JSON
    timeout        = 60,
)

print(f'\nJSON agora tem {len(registros_atualizados)} registro(s).')
iso_ok = [r for r in registros_atualizados if r.get("origem") == "obp-access"]
print(f'Registros vindos do obp-access: {len(iso_ok)}')


0 URL(s) ISO no CSV de erros.

0 norma(s) ISO adicionada(s) ao JSON.
0 ISO ainda em erro (registradas em paginas_com_erro.csv).

JSON agora tem 404 registro(s).
Registros vindos do obp-access: 0


## 11. Pós-processamento: limpar vazios e "lixo" do JSON coletado (para o RAG)

Mesmo com o isolamento de conteúdo central (seção 3.3), alguns resíduos passam:

- **Fragmentos técnicos do CMS** que vazam para o texto (ex.: `ID da página raiz: 43`,
  um campo de depuração de um plugin, visto no corpus da SBIS).
- **Blocos de navegação/menu repetidos** que escapam por não baterem com nenhum
  padrão de `_LIXO_PADROES` (ex.: uma lista de "páginas relacionadas" sem
  classe/id reconhecível — vimos isso se repetir *identicamente* em 4 das 5
  páginas do domínio `sbis.org.br`).
- **Figuras sem nenhuma informação útil** (`<svg>` decorativo sem `alt`, sem
  legenda e sem `url` — não ajuda em nada um RAG).
- **Páginas "cascas"**: title/URL existem, mas o conteúdo real ainda não foi
  publicado (ex.: uma página que, tirando o menu e o ID de depuração, só dizia
  "EM BREVE").

**Cuidado com corpora jurídicos.** Em `planalto.gov.br`, frases idênticas se
repetem *legitimamente* entre leis (citações cruzadas, redações dadas por leis
posteriores, o texto "compilado" vs. o texto original da mesma lei). Um
detector ingênuo de "texto repetido entre páginas" apagaria conteúdo jurídico
válido. Por isso a função abaixo:

1. só considera candidato um trecho que se repete em ≥ 2 páginas do MESMO domínio;
2. descarta automaticamente qualquer candidato que contenha marcadores de texto
   legal (`art.`, `lei nº`, `§`, `inciso`, `caput`, "(de 20xx)"...);
3. só **remove automaticamente** os candidatos restantes quando cobrem uma
   fração alta (≥ 60%) das páginas daquele domínio — padrão típico de
   menu/rodapé repetido em quase todas as páginas de um site;
4. os demais candidatos (repetição real, mas fora desses critérios de
   segurança) vão só para um **relatório de revisão manual**, sem apagar nada
   automaticamente.

Ao final, registros cujo texto fique abaixo de um tamanho mínimo útil (depois
da limpeza) são descartados — normalmente páginas "casca"/"em breve".

In [21]:
"""
Pós-processamento de conteudo_coletado.json para uso em RAG.
"""
import json, re
from collections import defaultdict, Counter
from urllib.parse import urlparse

# --- 1) Padrões de lixo técnico conhecido (vá completando esta lista conforme
#     forem aparecendo em novos sites) ---------------------------------------
_LIXO_TEXTO_REGEX = [
    r'\bID da p[aá]gina(?: raiz)?:\s*\d+\b',
]

# --- Limiares ----------------------------------------------------------------
MIN_TEXTO_UTIL_POS       = 1500   # abaixo disso, após a limpeza, o registro é descartado
MIN_TAMANHO_BLOCO        = 40    # nº mínimo de caracteres p/ um trecho virar candidato
MIN_PAGINAS_REPETICAO    = 2     # precisa repetir em pelo menos N páginas do domínio
FRACAO_MIN_AUTO_REMOCAO  = 0.3   # remove automaticamente só se cobrir >=60% das páginas do domínio

_MARCADORES_TEXTO_LEGAL = re.compile(
    r'\bart\.|\bartigo\b|\blei n[º°]|\bdecreto\b|§|\binciso\b|\bcaput\b|'
    r'\bpar[aá]grafo\b|\bmedida provis[oó]ria\b|\(de\s+\d{4}\)|de\s+\d{4}\)',
    re.IGNORECASE,
)


def _limpar_padroes_conhecidos(texto):
    for pat in _LIXO_TEXTO_REGEX:
        texto = re.sub(pat, ' ', texto, flags=re.IGNORECASE)
    return texto


def _candidatos_blocos_repetidos(registros):
    """Agrupa por domínio e conta em quantas páginas (URLs distintas) cada
    'frase' (trecho separado por pontuação forte) aparece."""
    por_dominio_trecho_urls = defaultdict(lambda: defaultdict(set))
    paginas_por_dominio = Counter()
    for r in registros:
        dom = urlparse(r['url']).netloc.lower()
        paginas_por_dominio[dom] += 1
        frases = re.split(r'(?<=[.!?;:])\s+', r['texto'])
        vistos_na_pagina = set()
        for f in frases:
            f = f.strip()
            if len(f) >= MIN_TAMANHO_BLOCO and f not in vistos_na_pagina:
                vistos_na_pagina.add(f)
                por_dominio_trecho_urls[dom][f].add(r['url'])
    return por_dominio_trecho_urls, paginas_por_dominio


def _classificar_candidatos(por_dominio_trecho_urls, paginas_por_dominio):
    """Separa candidatos em 'remover automaticamente' (alta confiança de ser
    menu/rodapé repetido) e 'revisar manualmente' (repete, mas não bate os
    critérios de segurança para remoção automática)."""
    auto_remover = defaultdict(set)
    revisar = []

    for dom, trechos in por_dominio_trecho_urls.items():
        total_paginas_dominio = paginas_por_dominio[dom]
        for trecho, urls in trechos.items():
            if len(urls) < MIN_PAGINAS_REPETICAO:
                continue
            eh_texto_legal = bool(_MARCADORES_TEXTO_LEGAL.search(trecho))
            fracao = len(urls) / total_paginas_dominio
            candidato = {
                'dominio': dom, 'trecho': trecho, 'n_paginas': len(urls),
                'total_paginas_dominio': total_paginas_dominio,
                'fracao': round(fracao, 2), 'parece_texto_legal': eh_texto_legal,
            }
            if not eh_texto_legal and fracao >= FRACAO_MIN_AUTO_REMOCAO:
                auto_remover[dom].add(trecho)
            else:
                revisar.append(candidato)

    return auto_remover, revisar


def limpar_registro(r, blocos_auto_remover, trechos_remover_manual=None):
    dom = urlparse(r['url']).netloc.lower()
    texto = _limpar_padroes_conhecidos(r['texto'])
    # 1) blocos detectados e removidos automaticamente (por domínio)
    for trecho in blocos_auto_remover.get(dom, ()):
        texto = texto.replace(trecho, ' ')
    # 2) trechos que VOCÊ escolheu remover da lista de revisão manual.
    #    Aplicados a todos os registros (não são específicos de um domínio), o que
    #    é seguro porque um texto só some se realmente contiver aquele trecho.
    for trecho in (trechos_remover_manual or ()):
        if trecho:
            texto = texto.replace(trecho, ' ')
    texto = ' '.join(texto.split())

    # Figuras sem NENHUMA informação útil (sem url, sem alt, sem legenda) não
    # ajudam um RAG e só ocupam espaço/ruído no dataset.
    figuras = [f for f in r['figuras'] if f.get('url') or f.get('alt') or f.get('legenda')]

    novo = dict(r)
    novo['texto'] = texto
    novo['figuras'] = figuras
    return novo


def pos_processar(caminho_entrada, caminho_saida, caminho_relatorio=None,
                  trechos_remover_manual=None):
    """Lê o JSON coletado, limpa e grava uma versão pronta para o RAG.

    Devolve (registros_limpos, registros_descartados, candidatos_para_revisao).
    """
    with open(caminho_entrada, 'r', encoding='utf-8') as f:
        dados = json.load(f)

    candidatos, paginas_por_dominio = _candidatos_blocos_repetidos(dados)
    auto_remover, revisar = _classificar_candidatos(candidatos, paginas_por_dominio)

    limpos, descartados = [], []
    figuras_removidas = 0
    for r in dados:
        novo = limpar_registro(r, auto_remover, trechos_remover_manual)
        figuras_removidas += len(r['figuras']) - len(novo['figuras'])
        if len(novo['texto'].strip()) < MIN_TEXTO_UTIL_POS:
            descartados.append({'url': novo['url'], 'motivo': 'texto_insuficiente_apos_limpeza',
                                  'tamanho': len(novo['texto'].strip())})
            continue
        limpos.append(novo)

    with open(caminho_saida, 'w', encoding='utf-8') as f:
        json.dump(limpos, f, ensure_ascii=False, indent=2)

    total_blocos_removidos = sum(len(v) for v in auto_remover.values())
    print(f'Entrada: {len(dados)} registro(s)')
    print(f'Saída (limpos): {len(limpos)} registro(s)')
    print(f'Descartados (texto insuficiente após limpeza): {len(descartados)}')
    print(f'Figuras vazias removidas: {figuras_removidas}')
    print(f'Blocos de navegação removidos automaticamente: {total_blocos_removidos}')
    for dom, trechos in auto_remover.items():
        for t in trechos:
            print(f'  - [{dom}] "{t[:100]}"')
    n_manuais = len([t for t in (trechos_remover_manual or ()) if t])
    if n_manuais:
        print(f'Trechos removidos manualmente (sua lista): {n_manuais}')
        for t in [t for t in trechos_remover_manual if t][:10]:
            print(f'  - "{t[:100]}"')
    print(f'Candidatos para revisão manual (NÃO removidos automaticamente): {len(revisar)}')
    print('  (veja o relatório completo para decidir se algum deles deve ser removido também)')

    if caminho_relatorio:
        with open(caminho_relatorio, 'w', encoding='utf-8') as f:
            json.dump({
                'descartados': descartados,
                'blocos_removidos_automaticamente': [
                    {'dominio': d, 'trecho': t} for d, ts in auto_remover.items() for t in ts
                ],
                'candidatos_para_revisao_manual': sorted(revisar, key=lambda x: -x['fracao'])[:200],
            }, f, ensure_ascii=False, indent=2)

    return limpos, descartados, revisar


print('Função de pós-processamento pronta.')


Função de pós-processamento pronta.


In [22]:
# Gera a versão limpa a partir do ARQUIVO_SAIDA já coletado.
ARQUIVO_SAIDA_LIMPO = ARQUIVO_SAIDA.replace('.json', '_limpo.json')
ARQUIVO_RELATORIO_LIMPEZA = 'relatorio_limpeza.json'

# ---------------------------------------------------------------------------
# Trechos que VOCÊ decidiu remover, escolhidos entre os "candidatos_para_revisao_
# manual" do relatorio_limpeza.json. Fluxo recomendado (duas passadas):
#
#   1ª passada: deixe a lista VAZIA e rode. Abra o relatorio_limpeza.json, veja
#               "candidatos_para_revisao_manual" e copie o campo "trecho" dos que
#               você quer eliminar (copie o texto EXATO, sem cortar).
#   2ª passada: cole esses trechos aqui e rode de novo — eles serão removidos do
#               texto de todos os registros que os contiverem.
#
# Dica: cole o trecho exatamente como está no relatório. A remoção é literal
# (str.replace); um registro só é afetado se realmente contiver aquele trecho.
# ---------------------------------------------------------------------------
TRECHOS_REMOVER_MANUAL = [
    # "cole aqui um trecho do relatório que queira remover",
    # "e outro trecho, se houver",
]

limpos, descartados, revisar = pos_processar(
    ARQUIVO_SAIDA,
    ARQUIVO_SAIDA_LIMPO,
    ARQUIVO_RELATORIO_LIMPEZA,
    trechos_remover_manual=TRECHOS_REMOVER_MANUAL,
)


Entrada: 404 registro(s)
Saída (limpos): 326 registro(s)
Descartados (texto insuficiente após limpeza): 78
Figuras vazias removidas: 40
Blocos de navegação removidos automaticamente: 8
  - [eur-lex.europa.eu] "An official website of the European Union An official EU website How do you know?"
  - [eur-lex.europa.eu] "This document is an excerpt from the EUR-Lex website Search tips Need more search options?"
  - [www.planalto.gov.br] "O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte Lei:"
  - [www.gov.br] "de segunda-feira a sexta-feira, das 8h às 20h, e aos sábados, das 8h às 18h."
  - [www.gov.br] "Membro da Vaccine Safety Net (VSN) Organização Mundial da Saúde – OMS O logotipo da VSN é de proprie"
  - [www.gov.br] "2º Esta Resolução entra em vigor na data de sua publicação."
  - [www.gov.br] "©2025 - Ministério da Saúde | Todos os direitos reservados."
  - [sbis.org.br] "Dimensões e domínios avaliados Estágios de maturidade Lista de requis